# 01 · data ingestion

pull a big universe of us etfs from `financedatabase` (it also gives us each fund's category, so we
dont have to hit yahoo per-ticker just to group them), download 5y of daily nav, drop the dead ones,
save to `data/raw/`.

change the exchanges / history length in `config.yaml`. `pip install financedatabase` if you dont have it.

In [ ]:
import yaml, pathlib, logging
import numpy as np, pandas as pd
import yfinance as yf
import financedatabase as fd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200, 'display.max_columns', 30)
logging.getLogger('yfinance').setLevel(logging.CRITICAL)   # quiet the 'delisted' spam

ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
cfg = yaml.safe_load(open(ROOT / 'config.yaml', encoding='utf-8'))
RAW = ROOT / cfg['paths']['raw']; RAW.mkdir(parents=True, exist_ok=True)
dcfg = cfg['data']

## build the universe

every etf on the exchanges we listed in config. financedatabase already carries name / category /
family so this is all local - no network yet.

In [ ]:
etfs = fd.ETFs().data
uni = etfs[etfs['exchange'].isin(cfg['exchanges'])].copy()
uni.index = uni.index.astype(str).str.strip()
uni = uni[~uni.index.duplicated()]
tickers = sorted(uni.index.unique())
print(len(tickers), 'etfs across', uni['category'].nunique(), 'categories')
uni['category'].value_counts().head(12)

## download prices

one batched pass over yahoo. delisted / dead tickers just wont return anything and we drop them.
~2k funds x 5y takes a few minutes.

In [ ]:
def download_closes(tickers, period, interval, auto_adjust, batch=300):
    out = []
    for i in range(0, len(tickers), batch):
        b = tickers[i:i+batch]
        df = yf.download(b, period=period, interval=interval, auto_adjust=auto_adjust,
                         progress=False, group_by='column', threads=True)
        if df.empty:
            continue
        if isinstance(df.columns, pd.MultiIndex):
            out.append(df['Close'])
        else:                                   # single-ticker batch comes back flat
            out.append(df[['Close']].set_axis(b, axis=1))
        print(f'  {min(i+batch, len(tickers)):>5}/{len(tickers)}')
    return pd.concat(out, axis=1)

prices = download_closes(tickers, dcfg['period'], dcfg['interval'], dcfg['auto_adjust'])
prices = prices.loc[:, ~prices.columns.duplicated()].sort_index()
prices.index.name = 'date'; prices.columns.name = 'ticker'
print('downloaded:', prices.shape)

## keep funds with enough history

anything with less than ~1y of data (config `min_days`) is too short to build 5d-ahead labels on.

In [ ]:
prices = prices.dropna(axis=1, how='all')
keep = prices.notna().sum() >= cfg['min_days']
prices = prices.loc[:, keep]
print(prices.shape[1], 'funds kept (>=', cfg['min_days'], 'days) |', prices.index.min().date(), '->', prices.index.max().date())
prices.iloc[-3:, :6]

## metadata + peer groups

straight from financedatabase - `category` is our peer group. no slow per-ticker yahoo calls.
(expense ratio / aum / holdings we can enrich later for a shortlist - too slow to pull for ~2k funds.)

In [ ]:
meta_cols = ['name','category_group','category','family','currency','exchange']
metadata = uni.reindex(prices.columns)[meta_cols].copy()
metadata.index.name = 'ticker'
metadata['category'] = metadata['category'].fillna('Uncategorized')
metadata['category_group'] = metadata['category_group'].fillna('Uncategorized')
print(metadata.shape)
metadata.head()

In [ ]:
# how many funds per category (this is what the label groups on)
sizes = metadata['category'].value_counts()
print(sizes.head(15))
print('\ncategories with <', cfg['labels']['min_group'], 'funds:', (sizes < cfg['labels']['min_group']).sum())

In [ ]:
# funds per category_group - a scannable view of what the universe holds
ax = metadata['category_group'].value_counts().plot.barh(figsize=(8, 6))
ax.set_title('funds per category group'); ax.invert_yaxis(); plt.tight_layout(); plt.show()

## save

In [ ]:
prices.to_parquet(RAW / 'prices.parquet');   prices.to_csv(RAW / 'prices.csv')
metadata.to_parquet(RAW / 'metadata.parquet'); metadata.to_csv(RAW / 'metadata.csv')
print('saved', prices.shape[1], 'funds to', RAW)